In [ ]:
import pandas as pd
import numpy as np
import re

with open("/home/cry_more/ongoing/LogRecall/data/HDFS.log",'r')as f:
    df = [line.strip() for line in f if line.strip()]

X = pd.DataFrame(df,columns=['logs'])
y = pd.read_csv("/home/cry_more/ongoing/LogRecall/data/anomaly_label.csv")

In [3]:
import re
import pandas as pd

# The fast regex pipeline
def mask_log(raw_log: str) -> str:
    cleaned = re.sub(r'^\s*\d+\s+\d+\s+\d+\s+[A-Z]+\s+[^:]+:\s*', '', raw_log)
    cleaned = re.sub(r'blk_-?\d+', '<BLOCK_ID>', cleaned)
    cleaned = re.sub(r'\b\d{1,3}(?:\.\d{1,3}){3}\b', '<IP>', cleaned)
    cleaned = re.sub(r'\b\d+\b', '<NUM>', cleaned)
    return cleaned.strip()

def process_in_chunks(input_file, output_file, chunk_size=100000):
    chunk_list = []
    
    # 'yield' makes this a generator. It streams the file line-by-line.
    with open(input_file, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            chunk_list.append(mask_log(line))
            
            # When we hit the chunk limit, dump to disk and clear RAM
            if (i + 1) % chunk_size == 0:
                df = pd.DataFrame({'Template': chunk_list})
                # Append to a CSV (or better, write to partitioned Parquet)
                df.to_csv(output_file, mode='a', header=not pd.io.common.file_exists(output_file), index=False)
                chunk_list.clear() # Free the memory instantly
                print(f"Processed {i + 1} lines...")
        
        # Process the final remainder
        if chunk_list:
            df = pd.DataFrame({'Template': chunk_list})
            df.to_csv(output_file, mode='a', header=not pd.io.common.file_exists(output_file), index=False)
            print("Finished.")

# Run it
process_in_chunks('/home/cry_more/ongoing/LogRecall/data/HDFS.log', 'HDFS_parsed.csv')

Processed 100000 lines...
Processed 200000 lines...
Processed 300000 lines...
Processed 400000 lines...
Processed 500000 lines...
Processed 600000 lines...
Processed 700000 lines...
Processed 800000 lines...
Processed 900000 lines...
Processed 1000000 lines...
Processed 1100000 lines...
Processed 1200000 lines...
Processed 1300000 lines...
Processed 1400000 lines...
Processed 1500000 lines...
Processed 1600000 lines...
Processed 1700000 lines...
Processed 1800000 lines...
Processed 1900000 lines...
Processed 2000000 lines...
Processed 2100000 lines...
Processed 2200000 lines...
Processed 2300000 lines...
Processed 2400000 lines...
Processed 2500000 lines...
Processed 2600000 lines...
Processed 2700000 lines...
Processed 2800000 lines...
Processed 2900000 lines...
Processed 3000000 lines...
Processed 3100000 lines...
Processed 3200000 lines...
Processed 3300000 lines...
Processed 3400000 lines...
Processed 3500000 lines...
Processed 3600000 lines...
Processed 3700000 lines...
Processed 

In [4]:
def fun(x):
    m = re.search(r"blk_", x)
    if m is None:
        return None
    st=m.start()
    en = x.find(" ",st)
    if en==-1:
        en = len(x)
    return x[st:en] 

X['BlockId'] = X['logs'].apply(lambda x:fun(x))
X = X.merge(y,on='BlockId',how='inner')

In [6]:
parsed_log = pd.read_csv("/home/cry_more/ongoing/LogRecall/data/HDFS_parsed.csv")
df = pd.concat([X["BlockId"],parsed_log],axis=True)

template_df = (
    df.groupby("BlockId", sort=False).agg(
        Template=("Template", "\n".join)
    ).reset_index()
)

In [ ]:
df = X.merge(y,on='BlockId',how='inner')
orig_df = (
    df.groupby('BlockId',sort=False).agg(
        logs=('logs','\n'.join),
        label=('Label','first')
    ).reset_index()
)

In [ ]:
template_df.to_parquet("/home/cry_more/ongoing/LogRecall/data/template_df.parquet",index=False)
orig_df.to_parquet("/home/cry_more/ongoing/LogRecall/data/orig_df.parquet",index=False)

ArrowKeyError: No type extension with name arrow.py_extension_type found